# USGS M4.5+ North American Earthquakes

Explore **3,742 significant earthquakes** (magnitude 4.5+) across North America.

**Dataset Highlights:**
- 25 years of data (2000-2025)
- Magnitude 4.5+ only
- North American coverage
- Removes 92.5% of minor tremors

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries loaded')

## 1. Load Dataset

In [ ]:
with open('usgs_significant_earthquakes.json') as f:
    data = json.load(f)

df = pd.DataFrame(data)
print(f'Total earthquakes: {len(df):,}')
df.head()

## 2. Magnitude Distribution

In [ ]:
valid_mag = df[df['magnitude'].notna()]

print('Magnitude Statistics:')
print('=' * 40)
print(f'Minimum: {valid_mag["magnitude"].min():.1f}')
print(f'Median: {valid_mag["magnitude"].median():.1f}')
print(f'Mean: {valid_mag["magnitude"].mean():.1f}')
print(f'Maximum: {valid_mag["magnitude"].max():.1f}')
print(f'\nMajor (M6+): {(valid_mag["magnitude"] >= 6).sum():,}')
print(f'Great (M7+): {(valid_mag["magnitude"] >= 7).sum():,}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(valid_mag['magnitude'], bins=30, color='darkred', edgecolor='black')
ax.set_xlabel('Magnitude')
ax.set_ylabel('Frequency')
ax.set_title('Earthquake Magnitude Distribution (M4.5+)', fontweight='bold')
ax.axvline(x=6.0, color='orange', linestyle='--', label='M6.0 (Major)')
ax.axvline(x=7.0, color='red', linestyle='--', label='M7.0 (Great)')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Largest Earthquakes

In [ ]:
largest = valid_mag.nlargest(15, 'magnitude')[['date', 'name', 'magnitude']]
print('Top 15 Largest Earthquakes:')
print('=' * 70)
for i, row in largest.iterrows():
    print(f"M{row['magnitude']:.1f}  {row['date'][:10]}  {row['name'][:50]}")

## 4. Geographic Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))
scatter = ax.scatter(df['longitude'], df['latitude'], 
                     c=valid_mag['magnitude'], cmap='YlOrRd',
                     alpha=0.6, s=valid_mag['magnitude']**2)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Earthquake Epicenters (size/color = magnitude)', fontweight='bold')
ax.set_xlim(-180, -60)
ax.set_ylim(15, 75)
plt.colorbar(scatter, label='Magnitude')
plt.tight_layout()
plt.show()

## 5. Temporal Patterns

In [ ]:
df['year'] = pd.to_datetime(df['date'], errors='coerce').dt.year
yearly = df.groupby('year').size()

fig, ax = plt.subplots(figsize=(12, 5))
yearly.plot(kind='bar', ax=ax, color='darkred', width=0.8)
ax.set_xlabel('Year')
ax.set_ylabel('Number of M4.5+ Earthquakes')
ax.set_title('Significant Earthquakes by Year', fontweight='bold')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 6. Regional Hotspots

In [ ]:
def categorize_region(lat, lon):
    if lat > 50 and lon < -130:
        return 'Alaska'
    elif lat < 25:
        return 'Caribbean/Mexico'
    elif lon < -115:
        return 'Pacific Coast'
    elif lon > -90:
        return 'Eastern US'
    else:
        return 'Central'

df['region'] = df.apply(lambda x: categorize_region(x['latitude'], x['longitude']), axis=1)
region_counts = df['region'].value_counts()

print('Earthquakes by Region:')
print('=' * 40)
for region, count in region_counts.items():
    pct = count / len(df) * 100
    print(f'{region:20s} {count:6,} ({pct:5.1f}%)')

## Conclusion

This notebook demonstrated:
- Loading 3,742 significant earthquakes
- Magnitude distribution analysis
- Geographic visualization of epicenters
- Temporal patterns over 25 years
- Regional hotspot identification

**Source**: USGS Earthquake Hazards Program

**Author**: Luke Steuber | @lukesteuber.com (Bluesky)